# Inherited Globe — Data Pipeline

Builds `sites.geojson` for the globe at `index.html`, from the official UNESCO World
Heritage List export in `data/whc001.csv`.

Four channels:

| # | Channel | What it gives |
|---|---|---|
| 1 | UNESCO World Heritage List export (CSV) | names, category, danger status, region, states, criteria, area, coordinates, components, photo gallery |
| 2 | Wikidata (SPARQL, property `P757`) | site ID → Wikipedia article title + `P18` image |
| 3 | Wikimedia Pageviews (REST) | 12-month human traffic per article → popularity |
| 4 | Wikipedia page summary (REST) | thumbnail, as an image fallback |

Only channel 1 is mandatory. Channels 2–4 hit public APIs and take roughly 30–60 min for
the full list; each has a checkpoint so a run can be resumed.

Run the cells top to bottom. The heavy cells are gated by the `RUN_*` flags in the
configuration cell.

---

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
# pip install pandas requests tqdm ipywidgets matplotlib
import importlib
import json
import sys
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

sys.path.insert(0, str(Path.cwd()))
from scripts import pipeline_helpers as ph

importlib.reload(ph)
pd.set_option("display.max_colwidth", 90)
print("helpers loaded from", ph.__file__)

---
### Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
WHC_CSV       = "data/whc001.csv"          # official UNESCO World Heritage List export
OUTPUT_PATH   = "sites.geojson"            # what index.html fetches
CACHE_DIR     = Path("data/cache")
CKPT_DIR      = CACHE_DIR / "checkpoints"
WIKIDATA_CACHE = CACHE_DIR / "wikidata" / "wikidata_map.json"
NAME_CACHE     = CACHE_DIR / "wikidata" / "name_fallback.json"
SECRETS_DIR   = Path("data/secrets")       # git-ignored

for folder in (CKPT_DIR, WIKIDATA_CACHE.parent, CACHE_DIR / "pageviews", CACHE_DIR / "thumbnails"):
    folder.mkdir(parents=True, exist_ok=True)

# ── Which network stages to run ───────────────────────────────────────────────
RUN_WIKIDATA_BATCH   = True   # channel 2, pass 1 — fast (a few SPARQL batches)
RUN_NAME_FALLBACK    = True   # channel 2, pass 2 — slow, one lookup per unresolved site
RUN_PAGEVIEWS        = True   # channel 3 — ~1 request/second per resolved article
RUN_WIKI_THUMBNAILS  = True   # channel 4 — only queried for sites with no UNESCO photo

# ── Label points ──────────────────────────────────────────────────────────────
# Serial properties publish one coordinate per component (up to 758 of them). Components
# are clustered by distance; the largest cluster always gets a point, secondary clusters
# only if they hold enough of the components.
COMPONENT_CLUSTER_BUFFER_KM = 250      # single-linkage distance between components
MAX_LABEL_POINTS_PER_SITE   = 1        # raise to 2 to also label big secondary clusters
SECONDARY_CLUSTER_MIN_SHARE = 0.25     # a secondary cluster needs >= 25% of components
JITTER_DUPLICATE_POINTS     = True     # nudge apart sites sharing exact coordinates

# ── Popularity ────────────────────────────────────────────────────────────────
# Last 12 completed months. Wikimedia's monthly endpoint expects YYYYMMDD.
PAGEVIEW_START = "20250601"
PAGEVIEW_END   = "20260531"
MIN_POPULARITY = 1   # sites with no article, or 0 views, still appear at the smallest size

# ── Popup content ─────────────────────────────────────────────────────────────
MAX_EXTRA_IMAGES     = 5    # extra UNESCO gallery photos kept as popup slides
DESCRIPTION_MAX_CHARS = 320 # short description trimmed before export
SHORT_LABEL_MAX_CHARS = 42  # on-globe label length before the full name is shortened

# ── Credits / versioning, shown in the globe's Method panel ───────────────────
WHC_EXPORT_NAME = "whc001"
WHC_EXPORT_DATE = "2026-09-16"   # the day data/whc001.csv was downloaded

# Optional: raises the Wikimedia rate limit from 500 to 5,000 req/hour.
# Create one at https://api.wikimedia.org/ and store it in the git-ignored file below.
WIKIMEDIA_TOKEN = ph.read_local_secret(SECRETS_DIR / "wikimedia_token.txt")

ph.configure(
    WIKIMEDIA_TOKEN=WIKIMEDIA_TOKEN,
    USER_AGENT="InheritedGlobe/1.0 (https://github.com/tdemareuil/inherited-globe)",
)
ph.set_pageview_window(PAGEVIEW_START, PAGEVIEW_END)

print("Wikimedia token:", "set" if WIKIMEDIA_TOKEN else "not set (500 req/hour)")
print("Pageview window:", PAGEVIEW_START, "→", PAGEVIEW_END)
print("Wikipedia language priority:", ", ".join(ph.WIKIPEDIA_LANGUAGE_PRIORITY[:10]), "…")

---
## 1. Load the UNESCO export

The export is the file published at
[whc.unesco.org/en/syndication](https://whc.unesco.org/en/syndication), one row per
inscribed property, in six UN languages. We keep the English fields plus the structured
ones, and drop the other five language blocks.

In [ ]:
raw = pd.read_csv(WHC_CSV, encoding="utf-8-sig")
print(f"{len(raw):,} rows × {raw.shape[1]} columns")
raw[["Name EN", "Catégorie", "Danger", "Region", "States Names", "Date inscribed"]].head()

In [ ]:
# Column presence check — the export's schema has changed across versions.
EXPECTED = [
    "Name EN", "Short Description EN", "Date inscribed", "Danger", "Danger list",
    "Area hectares", "Criteria", "Catégorie", "States Names", "ISO Codes", "Region",
    "Transboundary", "Main Image", "Main Image Author", "Main Image Copyright",
    "Images", "ID", "Coordonnées", "Components", "Components Count",
]
missing = [c for c in EXPECTED if c not in raw.columns]
print("missing columns:", missing or "none")

In [ ]:
def to_bool(value):
    """The export writes booleans as True/False, Y/N or 1/0 depending on the column."""
    text = ph.clean_str(value).lower()
    return text in {"true", "y", "yes", "1"}


sites = pd.DataFrame({
    "site_id":           raw["ID"].astype("Int64"),
    "label":             raw["Name EN"].map(ph.strip_tags),
    "short_description": raw["Short Description EN"].map(ph.strip_tags),
    "category":          raw["Catégorie"].map(lambda v: ph.clean_str(v).title()),
    "in_danger":         raw["Danger"].map(to_bool),
    "danger_list":       raw["Danger list"].map(ph.clean_str),
    "region":            raw["Region"].map(ph.clean_str),
    "states":            raw["States Names"].map(ph.clean_str),
    "iso_codes":         raw["ISO Codes"].map(lambda v: ph.clean_str(v).upper()),
    "date_inscribed":    pd.to_numeric(raw["Date inscribed"], errors="coerce").astype("Int64"),
    "criteria":          raw["Criteria"].map(ph.clean_str),
    "area_hectares":     pd.to_numeric(raw["Area hectares"], errors="coerce"),
    "transboundary":     raw["Transboundary"].map(to_bool),
    "component_count":   pd.to_numeric(raw["Components Count"], errors="coerce").fillna(0).astype(int),
    "_coordinates":      raw["Coordonnées"],
    "_components":       raw["Components"],
    "_main_image":       raw["Main Image"],
    "_images":           raw["Images"],
    "image_author":      raw["Main Image Author"].map(ph.clean_str),
    "image_copyright":   raw["Main Image Copyright"].map(ph.clean_str),
})

sites["short_label"] = sites["label"].map(lambda n: ph.short_label(n, SHORT_LABEL_MAX_CHARS))
sites["short_description"] = sites["short_description"].map(
    lambda t: t if len(t) <= DESCRIPTION_MAX_CHARS else t[:DESCRIPTION_MAX_CHARS].rsplit(" ", 1)[0] + "…"
)
sites["color_key"] = [
    ph.category_color_key(cat, danger)
    for cat, danger in zip(sites["category"], sites["in_danger"])
]
sites["unesco_url"] = sites["site_id"].map(ph.unesco_site_url)

print(sites["category"].value_counts().to_string())
print()
print("in danger:", int(sites["in_danger"].sum()))
print()
print(sites["region"].value_counts().to_string())
print()
unknown_regions = set(sites["region"]) - set(ph.UNESCO_REGIONS)
print("regions not in the index.html filter:", unknown_regions or "none")

In [ ]:
# Sanity: duplicate site IDs, empty names, unexpected categories
print("duplicate site IDs:", int(sites["site_id"].duplicated().sum()))
print("empty names:", int((sites["label"] == "").sum()))
print("unexpected categories:", set(sites["category"]) - set(ph.CATEGORIES) or "none")
print("date range:", int(sites["date_inscribed"].min()), "→", int(sites["date_inscribed"].max()))
sites[sites["short_label"].notna()][["label", "short_label"]].head(10)

---
## 2. Label points

Every property has one official coordinate pair. Serial properties also publish one
coordinate per component; those are clustered so a property split across a country
(the 758 components of the Frontiers of the Roman Empire, say) does not become 758 labels.

With `MAX_LABEL_POINTS_PER_SITE = 1` each property gets exactly one point, placed on its
largest cluster. Raise it to 2 and set `SHOW_MAIN_POINT_ONLY = false` in `index.html` to
let big secondary clusters carry a second label.

In [ ]:
records = []
no_location = []

for row in tqdm(sites.to_dict("records"), desc="Label points"):
    lat, lon = ph.parse_coordinates(row.pop("_coordinates"))
    components = ph.parse_components(row.pop("_components"))
    points = ph.site_label_points(
        lat, lon, components,
        buffer_km=COMPONENT_CLUSTER_BUFFER_KM,
        max_points=MAX_LABEL_POINTS_PER_SITE,
        secondary_min_share=SECONDARY_CLUSTER_MIN_SHARE,
    )
    if not points:
        no_location.append((row["site_id"], row["label"]))
        continue
    for rank, point in enumerate(points, start=1):
        record = dict(row)
        record.update({
            "lat": point["lat"],
            "lon": point["lon"],
            "point_source": point["point_source"],
            "cluster_component_count": point["cluster_component_count"],
            "cluster_share": point["cluster_share"],
            "label_rank": rank,
            "label_count": len(points),
        })
        records.append(record)

df = pd.DataFrame(records)
print(f"{df['site_id'].nunique():,} sites → {len(df):,} label points")
print(f"sites with more than one label point: {df.loc[df['label_count'] > 1, 'site_id'].nunique()}")
print()
print(f"{len(no_location)} site(s) with neither coordinates nor components — dropped:")
for site_id, label in no_location:
    print(f"  {site_id}  {label}")

In [ ]:
# Placement source breakdown
print(df["point_source"].value_counts().to_string())
print()
# Sites whose point came from clustering rather than the published coordinates
df[df["point_source"] == "component_cluster"][
    ["site_id", "label", "component_count", "cluster_component_count", "lat", "lon"]
].head(15)

In [ ]:
if JITTER_DUPLICATE_POINTS:
    df = ph.jitter_duplicate_points(df)

duplicates = df.groupby([df["lat"].round(6), df["lon"].round(6)]).size()
print("coordinate pairs still shared by 2+ label points:", int((duplicates > 1).sum()))

### Checkpoint 1 — after parsing & placement

In [ ]:
CKPT1 = CKPT_DIR / "ckpt_1_label_points.csv"
df.to_csv(CKPT1, index=False)
print("saved", CKPT1, df.shape)

In [ ]:
# ── Restart from checkpoint 1 ──
# df = pd.read_csv(CKPT1)
# print("restored", df.shape)

---
## 3. Wikipedia article resolution (channel 2)

Wikidata stores the UNESCO site ID as property
[`P757`](https://www.wikidata.org/wiki/Property:P757), so a single batched SPARQL query
maps most of the list to a Wikidata item, its Wikipedia sitelinks and its `P18` image —
the same shape as the IUCN `P627` lookup in the sister project.

Sites the query misses (mostly properties inscribed at the latest session, not yet in
Wikidata) go through a name-based fallback chain:

1. **Wikidata entity search** (`wbsearchentities`) on each variant of the English name.
2. **Direct Wikipedia title lookup**, resolving redirects.

Name variants drop a trailing parenthesised country, a leading `The `, an em-dash
subtitle, and UNESCO title prefixes such as `Historic Centre of `.

In [ ]:
site_ids = sorted(df["site_id"].dropna().astype(int).unique())
print(f"{len(site_ids):,} distinct site IDs to resolve")

wikidata_map = {}
if WIKIDATA_CACHE.exists():
    wikidata_map = json.loads(WIKIDATA_CACHE.read_text())
    print(f"loaded {len(wikidata_map):,} cached Wikidata entries from {WIKIDATA_CACHE}")

In [ ]:
if RUN_WIKIDATA_BATCH:
    pending = [i for i in site_ids if str(i) not in wikidata_map]
    print(f"querying Wikidata for {len(pending):,} sites")
    if pending:
        wikidata_map.update(ph.query_wikidata_batch(pending))
        WIKIDATA_CACHE.write_text(json.dumps(wikidata_map, ensure_ascii=False, indent=1))
        print(f"cached {len(wikidata_map):,} entries → {WIKIDATA_CACHE}")

resolved = {k for k, v in wikidata_map.items() if v.get("wiki_title")}
print(f"resolved by P757: {len(resolved):,} / {len(site_ids):,}")

In [ ]:
# What languages did the batch settle on?
if wikidata_map:
    languages = pd.Series([v.get("wiki_language") for v in wikidata_map.values()])
    print(languages.value_counts().head(12).to_string())

In [ ]:
# Name-based fallback for everything P757 did not resolve.
unresolved = [
    (int(r.site_id), r.label, r.states)
    for r in df.drop_duplicates("site_id").itertuples()
    if str(int(r.site_id)) not in resolved
]
print(f"{len(unresolved)} sites to resolve by name")
for site_id, label, _ in unresolved[:20]:
    print(f"  {site_id}  {label}")

In [ ]:
if RUN_NAME_FALLBACK and unresolved:
    name_map = ph.resolve_sites_by_name(unresolved, cache_path=NAME_CACHE)
    # Name-resolved entries never override a P757 hit.
    for key, entry in name_map.items():
        wikidata_map.setdefault(key, entry)
    WIKIDATA_CACHE.write_text(json.dumps(wikidata_map, ensure_ascii=False, indent=1))

still_missing = [
    int(r.site_id) for r in df.drop_duplicates("site_id").itertuples()
    if not (wikidata_map.get(str(int(r.site_id))) or {}).get("wiki_title")
]
print(f"still without a Wikipedia article: {len(still_missing)}")

In [ ]:
# Manual retry — force a specific article onto a site the chain could not resolve.
# Fill the dict below, re-run, then re-run the attach cell.
MANUAL_ARTICLES = {
    # 1567: "Funerary and memory sites of the First World War",
}
for site_id, title in MANUAL_ARTICLES.items():
    entry = ph.wikipedia_direct_search(title, lang="en")
    if entry:
        wikidata_map[str(site_id)] = entry
        print(f"  {site_id} → {entry['wiki_url']}")
    else:
        print(f"  {site_id}: no Wikipedia page for {title!r}")
if MANUAL_ARTICLES:
    WIKIDATA_CACHE.write_text(json.dumps(wikidata_map, ensure_ascii=False, indent=1))

In [ ]:
# Two sites resolving to the same article would share a popularity score.
# Keep it visible rather than silently deduplicating — some are genuine
# (transboundary inscriptions), some are a bad name match worth fixing above.
by_url = {}
for key, entry in wikidata_map.items():
    url = entry.get("wiki_url")
    if url:
        by_url.setdefault(url, []).append(key)
shared = {url: keys for url, keys in by_url.items() if len(keys) > 1}
print(f"{len(shared)} Wikipedia articles shared by more than one site")
names = df.drop_duplicates("site_id").set_index(df.drop_duplicates("site_id")["site_id"].astype(str))["label"]
for url, keys in list(shared.items())[:15]:
    print(f"  {url}")
    for key in keys:
        print(f"      {key}  {names.get(key, '?')}")

In [ ]:
df = ph.attach_wikidata_fields(df, wikidata_map)
print(df[["label", "wiki_title", "wiki_language", "wiki_lookup_source"]].head(10).to_string())
print()
print("with article:", int(df.drop_duplicates("site_id")["wiki_title"].notna().sum()),
      "/", df["site_id"].nunique())

### Checkpoint 2 — after Wikidata

In [ ]:
CKPT2 = CKPT_DIR / "ckpt_2_wikidata.csv"
df.to_csv(CKPT2, index=False)
print("saved", CKPT2, df.shape)

In [ ]:
# ── Restart from checkpoint 2 ──
# df = pd.read_csv(CKPT2)
# print("restored", df.shape)

---
## 4. Popularity — Wikipedia pageviews (channel 3)

Total human pageviews over the 12-month window, per resolved article. The `user` agent
filter excludes bots and crawlers.

Queried once per unique article, then filled back onto every label point of the site, so
a serial property with two points costs one request, not two.

Sites without an article, or whose article got no traffic, are set to
`MIN_POPULARITY = 1`: the globe sorts labels by `−popularity`, so a 0 would suppress them
entirely instead of showing them at the smallest size.

In [ ]:
articles = (
    df.dropna(subset=["wiki_title"])
      .drop_duplicates(subset=["wiki_project", "wiki_title"])[["wiki_project", "wiki_title"]]
)
print(f"{len(articles):,} unique articles to query (~{len(articles) * ph.SLEEP_PAGEVIEWS / 60:.0f} min)")

In [ ]:
import time

PAGEVIEWS_CACHE = CACHE_DIR / "pageviews" / f"pageviews_{PAGEVIEW_START}_{PAGEVIEW_END}.json"
pageviews = json.loads(PAGEVIEWS_CACHE.read_text()) if PAGEVIEWS_CACHE.exists() else {}
print(f"{len(pageviews):,} cached counts")

if RUN_PAGEVIEWS:
    pending = [
        (project, title) for project, title in articles.itertuples(index=False, name=None)
        if f"{project}|{title}" not in pageviews
    ]
    print(f"querying {len(pending):,} articles")
    for index, (project, title) in enumerate(tqdm(pending, desc="Pageviews"), start=1):
        pageviews[f"{project}|{title}"] = ph.get_pageviews(project, title)
        time.sleep(ph.SLEEP_PAGEVIEWS)
        if index % 100 == 0:
            PAGEVIEWS_CACHE.write_text(json.dumps(pageviews, ensure_ascii=False))
    PAGEVIEWS_CACHE.write_text(json.dumps(pageviews, ensure_ascii=False))
    print(f"cached {len(pageviews):,} counts → {PAGEVIEWS_CACHE}")

In [ ]:
keys = df["wiki_project"].fillna("") + "|" + df["wiki_title"].fillna("")
df["popularity"] = keys.map(pageviews).fillna(0).astype(int).clip(lower=MIN_POPULARITY)

floor = int((df.drop_duplicates("site_id")["popularity"] == MIN_POPULARITY).sum())
print(f"sites at the {MIN_POPULARITY}-view floor (no article, or no traffic): {floor}")
print()
top = df.drop_duplicates("site_id").nlargest(15, "popularity")
print(top[["label", "wiki_title", "popularity"]].to_string(index=False))

In [ ]:
# Sites on the floor despite having an article — usually a redirect title or a
# stub in a small-language edition. Worth a manual article override above.
suspect = df.drop_duplicates("site_id")
suspect = suspect[(suspect["popularity"] <= MIN_POPULARITY) & suspect["wiki_title"].notna()]
print(f"{len(suspect)} sites with an article but no views")
suspect[["site_id", "label", "wiki_project", "wiki_title"]].head(20)

### Checkpoint 3 — after pageviews

In [ ]:
CKPT3 = CKPT_DIR / "ckpt_3_pageviews.csv"
df.to_csv(CKPT3, index=False)
print("saved", CKPT3, df.shape)

In [ ]:
# ── Restart from checkpoint 3 ──
# df = pd.read_csv(CKPT3)
# print("restored", df.shape)

---
## 5. Images

The popup's photos come from the property's own UNESCO record: `Main Image` first, then
up to `MAX_EXTRA_IMAGES` more from the `Images` gallery, as extra slides. Author and
copyright come from the same row and are shown as the credit line.

Wikipedia and Wikidata images are appended as later slides, and become the primary image
for the handful of properties whose UNESCO record carries no photo.

> **Note.** UNESCO serves these photos from `whc.unesco.org/document/<id>` behind a bot
> challenge. Ordinary browsers load them; scripted checks from this notebook generally
> cannot, so image reachability is verified in the globe, not here.

In [ ]:
main_images  = raw.set_index(raw["ID"].astype(str))["Main Image"]
gallery      = raw.set_index(raw["ID"].astype(str))["Images"]
keys         = df["site_id"].astype(str)

def first_image(site_key):
    urls = ph.parse_url_list(main_images.get(site_key))
    return urls[0] if urls else None

def extra_images(site_key):
    primary = first_image(site_key)
    urls = [u for u in ph.parse_url_list(gallery.get(site_key)) if u != primary]
    return urls[:MAX_EXTRA_IMAGES]

df["image_url"]        = keys.map(first_image)
df["extra_image_urls"] = keys.map(lambda k: ", ".join(extra_images(k)))
df["image_source"]     = df["image_url"].map(lambda u: "UNESCO" if u else None)
df["image_credit"] = [
    " / ".join(dict.fromkeys(p for p in (author, copyright_) if p)) or None
    for author, copyright_ in zip(df["image_author"], df["image_copyright"])
]

unique = df.drop_duplicates("site_id")
print("sites with a UNESCO photo:", int(unique["image_url"].notna().sum()), "/", len(unique))
print("sites with extra gallery slides:", int((unique["extra_image_urls"] != "").sum()))
unique[unique["image_url"].isna()][["site_id", "label"]]

In [ ]:
# Wikipedia thumbnails — an extra slide for every site with an article, and the
# primary image for the few without a UNESCO photo. One request per unique article.
THUMBS_CACHE = CACHE_DIR / "thumbnails" / "wikipedia_thumbnails.json"
thumbnails = json.loads(THUMBS_CACHE.read_text()) if THUMBS_CACHE.exists() else {}
print(f"{len(thumbnails):,} cached thumbnails")

if RUN_WIKI_THUMBNAILS:
    pending = [
        (project, title) for project, title in articles.itertuples(index=False, name=None)
        if f"{project}|{title}" not in thumbnails
    ]
    print(f"querying {len(pending):,} thumbnails")
    for index, (project, title) in enumerate(tqdm(pending, desc="Thumbnails"), start=1):
        thumbnails[f"{project}|{title}"] = ph.get_wikipedia_thumbnail(project, title)
        time.sleep(ph.SLEEP_WIKI)
        if index % 200 == 0:
            THUMBS_CACHE.write_text(json.dumps(thumbnails, ensure_ascii=False))
    THUMBS_CACHE.write_text(json.dumps(thumbnails, ensure_ascii=False))
    print(f"cached {len(thumbnails):,} thumbnails → {THUMBS_CACHE}")

In [ ]:
article_keys = df["wiki_project"].fillna("") + "|" + df["wiki_title"].fillna("")
df["wikipedia_thumbnail_url"] = article_keys.map(thumbnails).map(ph.wikimedia_tiff_to_thumbnail)
df["wikidata_image_url"] = df["wikidata_image_url"].map(ph.wikimedia_tiff_to_thumbnail)

# Promote a Wikipedia/Wikidata image to primary where UNESCO has none.
fallback = df["image_url"].isna()
df.loc[fallback, "image_source"] = None
for column, source in (("wikipedia_thumbnail_url", "Wikipedia"), ("wikidata_image_url", "Wikidata")):
    take = fallback & df["image_url"].isna() & df[column].notna()
    df.loc[take, "image_url"] = df.loc[take, column]
    df.loc[take, "image_source"] = source
    df.loc[take, "image_credit"] = None

unique = df.drop_duplicates("site_id")
print(unique["image_source"].value_counts(dropna=False).to_string())
print()
print("sites with no image at all:", int(unique["image_url"].isna().sum()))
unique[unique["image_url"].isna()][["site_id", "label", "wiki_url"]]

In [ ]:
# Visual check — the first few UNESCO photos, as the popup will load them.
from IPython.display import HTML, display

sample = df.drop_duplicates("site_id")
sample = sample[sample["image_source"] == "UNESCO"].head(8)
display(HTML("".join(
    f'<figure style="display:inline-block;width:180px;margin:4px;vertical-align:top">'
    f'<img src="{row.image_url}" referrerpolicy="no-referrer" style="width:100%;border-radius:6px">'
    f'<figcaption style="font:11px sans-serif">{row.label[:60]}</figcaption></figure>'
    for row in sample.itertuples()
)))

---
## 6. Export

`sites.geojson` carries only the fields `index.html` reads. One feature per label point,
so a property with two label points appears twice, distinguished by `label_rank`.

In [ ]:
geojson = ph.build_geojson(df)
ph.write_geojson(geojson, OUTPUT_PATH)

example = max(geojson["features"], key=lambda f: f["properties"].get("popularity", 0))
print()
print(json.dumps(example, ensure_ascii=False, indent=2)[:1800])

In [ ]:
# Field coverage in the exported file
coverage = pd.Series({
    field: sum(1 for f in geojson["features"] if field in f["properties"])
    for field in ph.GEOJSON_FIELDS
})
total = len(geojson["features"])
pd.DataFrame({"present": coverage, "share": (coverage / total).round(3)})

---
## 7. Quality checks

Run these after an export to see what the globe will actually look like.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

popularity = df.drop_duplicates("site_id")["popularity"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

axes[0].hist(popularity.clip(upper=popularity.quantile(0.99)), bins=60, color="#A538F0")
axes[0].set_title("Pageviews per site (99th pct clipped)")

bins = np.logspace(0, np.log10(max(int(popularity.max()), 10)), 60)
axes[1].hist(popularity, bins=bins, color="#A538F0")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Pageviews per site (log-log)")

for ax in axes:
    ax.set_xlabel("views / year")
plt.tight_layout()
plt.show()

print(popularity.describe().round(0).to_string())

In [ ]:
# Coverage by category and region — what each filter combination will show
unique = df.drop_duplicates("site_id")
print(pd.crosstab(unique["region"], unique["color_key"], margins=True).to_string())

In [ ]:
# The 20 least-viewed sites: these only surface at high zoom
unique.nsmallest(20, "popularity")[["site_id", "label", "states", "popularity", "wiki_title"]]

In [ ]:
# Names still long enough to crowd the globe at low zoom
long_labels = unique.assign(display=unique["short_label"].fillna(unique["label"]))
long_labels = long_labels[long_labels["display"].str.len() > SHORT_LABEL_MAX_CHARS]
print(f"{len(long_labels)} labels longer than {SHORT_LABEL_MAX_CHARS} characters")
long_labels[["site_id", "label", "short_label"]].head(20)

In [ ]:
# Manual label override — set a shorter on-globe name for specific sites, then re-export.
LABEL_OVERRIDES = {
    # 86: "Pyramids of Giza",
}
for site_id, text in LABEL_OVERRIDES.items():
    df.loc[df["site_id"] == site_id, "short_label"] = text
if LABEL_OVERRIDES:
    ph.write_geojson(ph.build_geojson(df), OUTPUT_PATH)